# PCA — On lines and planes of closest fit / Principal Components (1901 / 1933)

_Papers · Classical ML_

**Find the few directions along which the data varies most; project onto them to compress with the least possible loss.**

---

This notebook builds PCA from scratch, one step at a time. Run each cell, read the note above it, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## Reference implementation — NumPy

We rebuild PCA from scratch on the classic 10-point toy cloud, then check it against scikit-learn. We go in four steps: (1) the PCA routine itself, (2) confirm it matches the `sklearn` oracle, (3) reconstruct from one component and tie the error to the discarded eigenvalue, (4) sweep the number of components kept.

### Step 1 — Write PCA from scratch

PCA finds the directions of greatest spread. The recipe is: center the cloud on its mean, form the covariance matrix $\Sigma = \frac{1}{n-1}X_c^\top X_c$, then eigen-decompose $\Sigma$. The eigenvectors are the principal directions and the eigenvalues are the variance captured along each. Because `np.linalg.eigh` returns eigenvalues in ascending order, we sort them descending so the first component is the largest.

In [ ]:
import numpy as np
from sklearn.decomposition import PCA

np.set_printoptions(precision=4, suppress=True)

# The classic toy 2-D set: 10 points, one row each.
X = np.array([[2.5, 2.4], [0.5, 0.7], [2.2, 2.9], [1.9, 2.2], [3.1, 3.0],
              [2.3, 2.7], [2.0, 1.6], [1.0, 1.1], [1.5, 1.6], [1.1, 0.9]])
n, d = X.shape


def my_pca(X, k):
    """PCA from scratch — Pearson(1901)/Hotelling(1933): center, covariance, eigen, project."""
    mean = X.mean(axis=0)                     # 1) centroid (the best line passes through it)
    Xc = X - mean                             #    center the cloud
    cov = (Xc.T @ Xc) / (X.shape[0] - 1)      # 2) covariance matrix Sigma (1/(n-1))
    w, V = np.linalg.eigh(cov)                # 3) eigen-decomp of symmetric Sigma
    order = np.argsort(w)[::-1]               #    eigh returns ascending -> sort descending
    w = w[order]
    V = V[:, order]
    components = V[:, :k].T                    # 4) top-k principal directions (rows)
    Z = Xc @ components.T                      #    project: scores (compressed coords)
    return mean, w, components, Z


mean, eigvals, comp, Z = my_pca(X, k=2)
print("mean:", mean)
print("eigenvalues (variance per PC):", eigvals)             # [1.2840 0.0491]
print("components (rows = PCs):\n", comp)
print("explained variance ratio:", eigvals / eigvals.sum())  # [0.9632 0.0368]

### Step 2 — Check against the sklearn oracle

A from-scratch implementation is only trustworthy if it matches a reference. We fit `sklearn.decomposition.PCA` on the same data and compare. One subtlety: an eigenvector's sign is arbitrary ($u$ and $-u$ describe the same line), so we compare the *magnitudes* of the components, while the eigenvalues should match exactly.

In [ ]:
# THE ORACLE: must match sklearn.decomposition.PCA.
skl = PCA(n_components=2).fit(X)

# Eigenvector sign is arbitrary (u and -u are the same direction) -> compare magnitudes.
comp_match = np.allclose(np.abs(comp), np.abs(skl.components_), atol=1e-6)
var_match = np.allclose(eigvals, skl.explained_variance_, atol=1e-6)
print("components match sklearn (up to sign):", comp_match)   # True
print("explained variance matches sklearn:    ", var_match)   # True

### Step 3 — Reconstruct from one component

Keeping only the top component, we project each point down to 1-D and then lift it back into the original 2-D space. The mean squared reconstruction error measures how much we lost. The key identity: that error equals the *discarded* eigenvalue scaled by $(n-1)/n$ — the variance you threw away is exactly the error you pay.

In [ ]:
# Variance-maximizing projection + reconstruction (k=1).
mean1, w1, comp1, Z1 = my_pca(X, k=1)
recon1 = Z1 @ comp1 + mean1                            # back to original space
mse1 = np.mean(np.sum((X - recon1) ** 2, axis=1))      # mean squared reconstruction error

dropped = eigvals[1] * (n - 1) / n                     # the variance we discarded
print("k=1 reconstruction MSE:", round(mse1, 4))               # 0.0442
print("dropped eigenvalue * (n-1)/n:", round(dropped, 4))      # 0.0442 -> error == discarded variance

### Step 4 — Sweep the number of components (ablation)

Finally we vary $k$ from 0 to 2 and watch the reconstruction error fall. With $k=0$ every point collapses to the mean (worst case); with $k=2$ we keep both directions and reconstruct perfectly. Each component added removes the next eigenvalue from the error, so the curve drops fast then flattens — the "elbow" that guides how many components to keep.

In [ ]:
# Ablation: reconstruction error vs number of components kept.
for k in [0, 1, 2]:
    Xc = X - mean
    if k == 0:
        recon = np.tile(mean, (n, 1))            # keep nothing: reconstruct as the mean
    else:
        Uk = comp[:k].T                          # top-k eigenvectors as columns
        recon = (Xc @ Uk) @ Uk.T + mean
    err = np.mean(np.sum((X - recon) ** 2, axis=1))
    print(f"k={k}: reconstruction MSE = {err:.4f}")  # 1.1998, 0.0442, 0.0000

## Visualize the data & results

_On a toy 2-D cloud that lies along a diagonal, how much variance does each principal component capture, and how does reconstruction error fall as we keep more components (the ablation)?_

### Step — Recompute the components, then sweep k

This panel reproduces the explained-variance ratio and the reconstruction-error-vs-$k$ curve directly from the eigen-decomposition. We first rebuild the centered covariance and its sorted eigenvectors, then loop over $k$ to print the error at each level — the curve you would plot to spot the elbow.

In [ ]:
import numpy as np

X = np.array([[2.5, 2.4], [0.5, 0.7], [2.2, 2.9], [1.9, 2.2], [3.1, 3.0],
              [2.3, 2.7], [2.0, 1.6], [1.0, 1.1], [1.5, 1.6], [1.1, 0.9]])
n = X.shape[0]

mean = X.mean(axis=0)
Xc = X - mean
cov = (Xc.T @ Xc) / (n - 1)
w, V = np.linalg.eigh(cov)
order = np.argsort(w)[::-1]
w = w[order]
V = V[:, order]

print("explained variance ratio:", np.round(w / w.sum(), 4))   # [0.9632 0.0368]

for k in [0, 1, 2]:                                            # ablation: error vs k
    if k == 0:
        recon = np.tile(mean, (n, 1))
    else:
        Uk = V[:, :k]
        recon = (Xc @ Uk) @ Uk.T + mean
    err = np.mean(np.sum((X - recon) ** 2, axis=1))
    print(f"k={k}: reconstruction MSE = {err:.4f}")            # 1.1998, 0.0442, 0.0000

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** For the toy set, the eigenvalues are $\lambda_1=1.2840$ and $\lambda_2=0.0491$. What fraction of the total variance does the first principal component explain, and what does that say about reducing the data to 1-D?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Total variance $=\lambda_1+\lambda_2=1.2840+0.0491=1.3331$. — _Total variance is the sum of all eigenvalues (the trace of $\Sigma$)._
- Fraction $=\lambda_1/1.3331=0.963$. — _Explained-variance ratio of component 1._

**Answer:** The first component explains about 96.3% of the variance. Dropping to one dimension throws away only ~3.7% of the spread, so a 1-D summary is almost lossless here — exactly why PCA is good at compression when the data is stretched along a few directions.

</details>

**Problem 2.** Show why minimizing perpendicular reconstruction error is the same as maximizing projected variance.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- For a unit direction $u$, split each centred point: $\|\tilde x_i\|^2=(\tilde x_i^\top u)^2+\|\tilde x_i-(\tilde x_i^\top u)u\|^2$. — _Pythagoras: projection plus perpendicular leftover._
- Sum over all points: total variance $=$ captured variance $+$ perpendicular error. — _The left side does not depend on $u$._
- So increasing captured variance must decrease perpendicular error by the same amount. — _Their sum is constant._

**Answer:** Because total variance is fixed, captured variance and perpendicular error trade off one-for-one. Maximizing $u^\top\Sigma u$ (Hotelling) therefore minimizes $\sum\|\tilde x_i-(\tilde x_i^\top u)u\|^2$ (Pearson). They are literally the same optimization, which is why both papers land on the leading eigenvector of $\Sigma$.

</details>

**Problem 3.** Ablation: keep $k=0$, then $k=1$, then $k=2$ components and measure the mean squared reconstruction error on the toy set. Predict the three numbers from the eigenvalues and confirm the trend.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- $k=0$: reconstruct every point as the mean. Error $=$ total variance $\cdot\frac{n-1}{n}=1.3331\cdot 0.9=1.1998$. — _Keeping nothing pays the whole variance._
- $k=1$: error $=\lambda_2\cdot\frac{n-1}{n}=0.0491\cdot0.9=0.0442$. — _You discard only the second direction's variance._
- $k=2$: error $=0$. — _You kept both directions — perfect reconstruction in 2-D._

**Answer:** Error falls 1.1998 → 0.0442 → 0 as $k$ goes 0 → 1 → 2. Each step removes the next-largest eigenvalue from the error, so the curve drops fast then flattens — the "elbow" you look for when choosing $k$. In CODEVIZ we plot exactly this reconstruction-error-vs-$k$ curve (our small run, not a paper number).

</details>